线性 SVM + HOG → 系数热力图

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import hog
from skimage.transform import resize
from sklearn.svm import LinearSVC

def hog_coefficient_heatmap(img, svm_model, hog_params=None):
    """
    根据线性 SVM 的 coef_ 生成 HOG 权重热力图（大小与原图相同）
    
    img: (H, W, 3) numpy array, uint8
    svm_model: 已训练的 sklearn LinearSVC
    hog_params: dict, 传给 skimage.feature.hog 的参数
                须包含 orientations, pixels_per_cell, cells_per_block
    """
    if hog_params is None:
        hog_params = dict(orientations=9, pixels_per_cell=(8, 8),
                          cells_per_block=(2, 2), block_norm='L2-Hys',
                          channel_axis=-1)
    
    # 获取 SVM 权重 (仅限线性核)
    if not hasattr(svm_model, 'coef_'):
        raise ValueError("仅支持线性 SVM")
    weights = svm_model.coef_.flatten()  # 长度为 HOG 描述符维度
    
    # 计算原图下 HOG 网格尺寸
    # HOG 输出形状: (n_cells_row, n_cells_col, n_blocks_row, n_blocks_col, n_orientations)
    from skimage.feature import _hog_channel_gradient
    # 简便方法: 用 hog 函数获取网格数
    fd, hog_image = hog(img, visualize=True, **hog_params)
    # hog_image 是每个单元格的平均梯度强度, 形状 (n_cells_row, n_cells_col)
    
    # 更精确: 按权重分配到每个 cell 块 (每个 block 内 cell 共享权重)
    # 这里采用粗略方案: 对每个 cell 累加其相关权重的绝对值
    orientations = hog_params['orientations']
    px = hog_params['pixels_per_cell'][0]
    py = hog_params['pixels_per_cell'][1]
    cb = hog_params['cells_per_block'][0]  # 假设 square block
    
    n_cells_row = (img.shape[0] // py) - cb + 1
    n_cells_col = (img.shape[1] // px) - cb + 1
    
    # 将 weights 重新组织为 (n_cells_row, n_cells_col, cb, cb, orientations)
    weight_map = weights.reshape(n_cells_row, n_cells_col, cb, cb, orientations)
    # 对方向求和, 对 block 内 cell 求和, 得到每个 cell 的总重要性
    cell_importance = np.sum(np.abs(weight_map), axis=(2, 3, 4))  # (n_cells_row, n_cells_col)
    
    # 上采样到原图大小
    heatmap = resize(cell_importance, img.shape[:2], order=1, mode='reflect')
    # 归一化到 [0, 1]
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
    return heatmap


def save_and_display_hog_heatmap(img, heatmap, save_path="hog_heatmap.jpg", alpha=0.4):
    """与原始 save_and_display_gradcam 接口一致"""
    import matplotlib.cm as cm
    
    heatmap_uint8 = np.uint8(255 * heatmap)
    jet = plt.colormaps['jet']
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap_uint8]
    
    # 改成与原图相同大小 (已经是了)
    img_float = img.astype(np.float32)
    if img_float.max() > 1.0:
        img_float /= 255.0
    
    superimposed = jet_heatmap * alpha + img_float
    superimposed = np.clip(superimposed, 0, 1)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(superimposed)
    ax.axis('off')
    ax.set_title('SVM + HOG Explanation (Linear Weights)')
    
    norm = plt.Normalize(vmin=0, vmax=1)
    sm = plt.cm.ScalarMappable(cmap='jet', norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('HOG Cell Importance', rotation=270, labelpad=15)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

使用方法

In [ ]:
# 训练线性 SVM + HOG
from sklearn.svm import LinearSVC
# ... 训练代码省略, 得到 model_svm
# 读取测试图片
sample_img = load_img(sample_path)  # 返回 PIL Image
img_np = np.array(sample_img)       # (H, W, 3) uint8

heatmap = hog_coefficient_heatmap(img_np, model_svm)
save_and_display_hog_heatmap(img_np, heatmap)

随机森林 + LBP → LIME 热力图

In [ ]:
import lime
from lime import lime_image
from skimage.segmentation import mark_boundaries

def explain_with_lime(img_np, classifier_fn, positive_label=1):
    """
    img_np: (H, W, 3) uint8
    classifier_fn: 函数, 输入 batch (N, H, W, 3), 输出预测概率 (N,)
    """
    explainer = lime_image.LimeImageExplainer()
    explanation = explainer.explain_instance(
        img_np,
        classifier_fn,
        top_labels=1,
        hide_color=0,
        num_samples=1000
    )
    # 获取正类（病灶）的掩膜
    temp, mask = explanation.get_image_and_mask(
        positive_label,
        positive_only=True,
        num_features=5,
        hide_rest=False
    )
    # 生成热力图: 用超像素 mask 的权重
    weight_map = np.zeros(img_np.shape[:2], dtype=np.float32)
    for feat, w in explanation.local_exp[positive_label]:
        # feat 是超像素 id, w 是权重
        weight_map[mask == feat] = w
    # 归一化
    weight_map = (weight_map - weight_map.min()) / (weight_map.max() - weight_map.min() + 1e-8)
    return weight_map

def save_and_display_lime(img_np, heatmap, save_path="lime_heatmap.jpg", alpha=0.4):
    """绘制 LIME 热力图叠加"""
    import matplotlib.cm as cm
    
    heatmap_uint8 = np.uint8(255 * heatmap)
    jet = plt.colormaps['jet']
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap_uint8]
    
    img_float = img_np.astype(np.float32) / 255.0
    superimposed = jet_heatmap * alpha + img_float
    superimposed = np.clip(superimposed, 0, 1)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(superimposed)
    ax.axis('off')
    ax.set_title('RF + LBP Explanation via LIME')
    
    norm = plt.Normalize(vmin=0, vmax=1)
    sm = plt.cm.ScalarMappable(cmap='jet', norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Superpixel Importance', rotation=270, labelpad=15)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

使用方法

In [ ]:
import cv2
from skimage.feature import local_binary_pattern

def lbp_rf_predict(images_batch):
    '''images_batch: (N, H, W, 3) uint8, 返回 (N,) 的概率'''
    probs = []
    for img in images_batch:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')
        # 生成 LBP 直方图特征
        hist, _ = np.histogram(lbp, bins=10, range=(0, 10))
        prob = rf_model.predict_proba([hist])[0, 1]  # 正类概率
        probs.append(prob)
    return np.array(probs)

# 使用
heatmap = explain_with_lime(img_np, lbp_rf_predict)
save_and_display_lime(img_np, heatmap)